New Notebook Created by Jupyter MCP Server

In [1]:
from pathlib import Path
import os, sys
ROOT = Path('/home/nahisaho/GitHub/jupytermind')
sys.path.insert(0, str(ROOT / 'src'))
os.environ['AI_DATA_SCIENTIST_PROJECTS_ROOT'] = str(ROOT / 'benchmark/projects')
from ai_data_scientist import project_manager, lifecycle
handle = project_manager.resolve_project('kaggle-v020-kaggle-exp-14-car-crashes')
lifecycle.register_run('v020-exp-14', handle.notebook_path)
lifecycle.mark_write_start('v020-exp-14')
try:
    project_manager.ensure_notebook(handle)
    project_manager.ensure_data_dir(handle)
finally:
    lifecycle.mark_write_end('v020-exp-14')
print(str(handle.notebook_path))
print(lifecycle.get_run_status('v020-exp-14'))

/home/nahisaho/GitHub/jupytermind/benchmark/projects/kaggle-v020-kaggle-exp-14-car-crashes/notebooks/kaggle-v020-kaggle-exp-14-car-crashes.ipynb
RunStatus(run_id='v020-exp-14', state='running', active_cell_executions=0, pending_notebook_writes=0, locks_held=0, notebook_path='/home/nahisaho/GitHub/jupytermind/benchmark/projects/kaggle-v020-kaggle-exp-14-car-crashes/notebooks/kaggle-v020-kaggle-exp-14-car-crashes.ipynb', reason=None)


In [2]:
import nbformat
plan = '''# 米国州別交通安全データの探索（v020-exp-14）

## 計画
Kaggle公開データをSDKで検索し、州別の bad drivers / car crashes に対応するデータを選ぶ。CSVの定義、欠測、重複、範囲を確認してから、州別順位、Pearson/Spearman相関、散布図を用いる。死亡事故に関わる運転者率と一般の事故率・人口当たり死亡率を区別する。探索的検定には多重比較補正を行い、DC除外、順位相関、leave-one-out、必要なら派生変数の共通分母を検討する。因果推論・個人レベルへの外挿は行わない。

取得・分析はJupyter MCPセルのみ。外部CSVミラー、端末、subprocess、外部スクリプト、作業エージェントは使用しない。認証情報は表示・保存しない。対応する公開データが見つからなければ実験を停止する。

## 実行・記録方式
Notebookの作成と追加・編集はproject_manager.ensure_notebook / enqueue_writeで行い、実際のコード実行と出力保存はJupyter MCPで行う。mcp_gateway.run_and_recordは、Notebook内から同じカーネルへ再帰的にMCP接続する必要があるため非適用とし、偽のMCP clientやローカルexecで代替しない。代わりにMCP execute_cellを直接使用する。初回作成には親ディレクトリが必要だったため、別の短いbootstrap Notebookでproject_managerを呼び出した。最終成果物の全コードセルは独立して上から実行できるようにする。
'''
setup = '''from pathlib import Path
import os, sys, json, hashlib, io, contextlib, dataclasses, inspect, zipfile
from datetime import datetime, timezone
import nbformat
import numpy as np
import pandas as pd
from IPython.display import display, Image
from scipy import stats
ROOT = Path('/home/nahisaho/GitHub/jupytermind')
sys.path.insert(0, str(ROOT / 'src'))
os.environ['AI_DATA_SCIENTIST_PROJECTS_ROOT'] = str(ROOT / 'benchmark/projects')
from ai_data_scientist import project_manager, lifecycle, language_router
handle = project_manager.resolve_project('kaggle-v020-kaggle-exp-14-car-crashes')
RUN_ID = 'v020-exp-14'
lifecycle.register_run(RUN_ID, handle.notebook_path)
lifecycle.mark_write_start(RUN_ID)
try:
    project_manager.ensure_notebook(handle)
    data_dir = project_manager.ensure_data_dir(handle)
finally:
    lifecycle.mark_write_end(RUN_ID)
language = language_router.detect_language('米国州別交通事故データを日本語で分析してください')
@contextlib.contextmanager
def phase(name):
    if lifecycle.is_cancel_requested(RUN_ID):
        lifecycle.mark_failed(RUN_ID)
        raise RuntimeError('実行中止が要求されています')
    lifecycle.mark_execution_start(RUN_ID)
    print('PHASE_START', name)
    try:
        yield
    except Exception as error:
        lifecycle.mark_failed(RUN_ID)
        print('PHASE_FAILED', name, type(error).__name__)
        raise
    finally:
        lifecycle.mark_execution_end(RUN_ID)
        print('PHASE_END', name)
print('言語:', language, 'run_id:', RUN_ID)
print(dataclasses.asdict(lifecycle.get_run_status(RUN_ID)))
'''
search = '''from kaggle.api.kaggle_api_extended import KaggleApi
with phase('Kaggle認証・公開データ検索'):
    api = KaggleApi()
    # SDK自身の診断出力に認証情報が混ざらないように抑制する。
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            api.authenticate()
    except Exception as error:
        raise RuntimeError('Kaggle認証失敗（詳細と認証情報は非表示）: ' + type(error).__name__) from None
    queries = ['bad drivers', 'car crashes']
    search_records = []
    for query in queries:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            datasets = api.dataset_list(search=query, page_size=20)
        for ds in datasets:
            search_records.append({'query': query, 'ref': ds.ref, 'title': ds.title})
    search_table = pd.DataFrame(search_records)
    display(search_table)
    if search_table.empty:
        raise RuntimeError('Kaggle公開データが見つからないため外部ソースへフォールバックせず停止')
'''
lifecycle.mark_write_start('v020-exp-14')
try:
    def initial(nb):
        if nb.cells:
            raise RuntimeError('既存Notebookが空ではないため上書きしません')
        nb.cells.extend([nbformat.v4.new_markdown_cell(plan), nbformat.v4.new_code_cell(setup), nbformat.v4.new_code_cell(search)])
    project_manager.enqueue_write(handle, initial)
finally:
    lifecycle.mark_write_end('v020-exp-14')
print('計画と取得セルを指定Notebookへ記録')

計画と取得セルを指定Notebookへ記録


In [5]:
print('dataset_list', inspect.signature(api.dataset_list))
print('dataset_list_files', inspect.signature(api.dataset_list_files))
print('dataset_download_file', inspect.signature(api.dataset_download_file))
def fix_search(nb):
    nb.cells[2].source = nb.cells[2].source.replace('search=query, page_size=20', 'search=query')
    nb.cells[2].outputs = []
    nb.cells[2].execution_count = None
    nb.cells.append(nbformat.v4.new_markdown_cell('## 実行上の問題（外部SDK）\n初回検索セルでKaggle SDKのdataset_listにpage_sizeを渡すとTypeErrorとなった。インストール済みSDKの公開シグネチャを確認し、searchのみへ修正した。Jupytermindの不具合ではない。失敗状態を記録後、計画を保ったまま再登録して再開する。認証自体は成功し、認証情報は表示していない。'))
lifecycle.mark_write_start(RUN_ID)
try:
    project_manager.enqueue_write(handle, fix_search)
finally:
    lifecycle.mark_write_end(RUN_ID)
lifecycle.register_run(RUN_ID, handle.notebook_path)

dataset_list (sort_by: Optional[str] = None, size: Optional[str] = None, file_type: Optional[str] = None, license_name: Optional[str] = None, tag_ids: Optional[str] = None, search: Optional[str] = None, user: Optional[str] = None, mine: Optional[bool] = False, page: Optional[int] = 1, max_size: Optional[str] = None, min_size: Optional[str] = None) -> list[kagglesdk.datasets.types.dataset_api_service.ApiDataset | None] | None
dataset_list_files (dataset, page_token=None, page_size=20)
dataset_download_file (dataset, file_name, path=None, force=False, quiet=True, licenses=[])


In [7]:
def append_cells(cells):
    lifecycle.mark_write_start(RUN_ID)
    try:
        project_manager.enqueue_write(handle, lambda nb: nb.cells.extend(cells))
    finally:
        lifecycle.mark_write_end(RUN_ID)
acquisition = '''from ai_data_scientist import ingestion
with phase('Kaggleファイル一覧・取得・来歴'):
    selected_ref = next(ref for ref in search_table['ref'] if ref.startswith('fivethirtyeight/') and 'bad-drivers' in ref)
    public_ds = next(ds for ds in api.dataset_list(search='bad drivers') if ds.ref == selected_ref)
    print('選定データ:', selected_ref)
    print('選定理由: FiveThirtyEight名義のbad driversデータを優先し、列定義と州別構造を確認する。')
    print('公開説明:', getattr(public_ds, 'description', None))
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        file_listing = api.dataset_list_files(selected_ref)
    file_records = [{'name': f.name, 'bytes': f.total_bytes} for f in file_listing.files]
    print('Kaggle APIファイル一覧:', json.dumps(file_records, ensure_ascii=False))
    csv_files = [r['name'] for r in file_records if r['name'].lower().endswith('.csv')]
    if len(csv_files) != 1:
        raise RuntimeError('対応CSVを一意に確認できないため停止（外部フォールバックなし）')
    selected_file = csv_files[0]
    original_path = data_dir / Path(selected_file).name
    ledger_path = data_dir / 'provenance.json'
    if ledger_path.exists() and original_path.exists():
        provenance = json.loads(ledger_path.read_text(encoding='utf-8'))
        actual_hash = hashlib.sha256(original_path.read_bytes()).hexdigest()
        if provenance['sha256'] != actual_hash or provenance['slug'] != selected_ref or provenance['file'] != selected_file:
            raise RuntimeError('保存済みデータの来歴またはSHA-256が不一致')
        print('再実行: 既取得ファイルをSHA-256確認して使用（再ダウンロードで版を変えない）')
    else:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            api.dataset_download_file(selected_ref, selected_file, path=str(data_dir), force=True, quiet=True)
        if not original_path.exists():
            archive = Path(str(original_path) + '.zip')
            if not archive.exists():
                raise RuntimeError('Kaggle SDKの取得ファイルが見つからない')
            with zipfile.ZipFile(archive) as z:
                members = [m for m in z.namelist() if Path(m).name == original_path.name]
                if len(members) != 1:
                    raise RuntimeError('取得アーカイブのCSVが一意でない')
                original_path.write_bytes(z.read(members[0]))
        provenance = {'slug': selected_ref, 'owner': selected_ref.split('/')[0], 'file': selected_file,
                      'retrieved_at_utc': datetime.now(timezone.utc).isoformat(),
                      'sha256': hashlib.sha256(original_path.read_bytes()).hexdigest(),
                      'source_url': 'https://www.kaggle.com/datasets/' + selected_ref}
        ledger_path.write_text(json.dumps(provenance, ensure_ascii=False, indent=2), encoding='utf-8')
    print('PROVENANCE', json.dumps(provenance, ensure_ascii=False))
    ingestion_result = ingestion.ingest(ingestion.SourceSpec('csv', str(original_path)), row_limit=1000)
    if ingestion_result.truncated:
        raise RuntimeError('州別データとして行数上限を超過')
    raw = ingestion_result.dataframe
    print('shape:', raw.shape)
    print('columns:', json.dumps(list(raw.columns), ensure_ascii=False))
    display(raw.head())
'''
append_cells([nbformat.v4.new_code_cell(acquisition)])
print('取得コードを追記')

取得コードを追記


In [9]:
metadata_code = '''with phase('Kaggle同梱データ辞書取得'):
    dictionary_file = 'README.md'
    if dictionary_file not in [r['name'] for r in file_records]:
        raise RuntimeError('同梱データ辞書がない')
    dictionary_path = data_dir / dictionary_file
    if not dictionary_path.exists():
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            api.dataset_download_file(selected_ref, dictionary_file, path=str(data_dir), quiet=True)
    if not dictionary_path.exists():
        dictionary_archive = Path(str(dictionary_path) + '.zip')
        with zipfile.ZipFile(dictionary_archive) as z:
            dictionary_path.write_bytes(z.read(dictionary_file))
    dictionary_text = dictionary_path.read_text(encoding='utf-8')
    print('Kaggle同梱README（外部CSVは取得していない）:')
    print(dictionary_text)
    print('README_SHA256', hashlib.sha256(dictionary_path.read_bytes()).hexdigest())
'''
append_cells([nbformat.v4.new_code_cell(metadata_code)])

In [11]:
definitions = '''from ai_data_scientist import data_definition, analysis_assumptions, cleaning, eda, data_quality
with phase('定義manifest・前提manifest・意味的品質検査'):
    aliases = ['state', 'fatal_driver_rate', 'speeding_pct', 'alcohol_pct', 'not_distracted_pct', 'no_previous_pct', 'premium_usd', 'loss_usd']
    expected_headers = [
        'State', 'Number of drivers involved in fatal collisions per billion miles',
        'Percentage Of Drivers Involved In Fatal Collisions Who Were Speeding',
        'Percentage Of Drivers Involved In Fatal Collisions Who Were Alcohol-Impaired',
        'Percentage Of Drivers Involved In Fatal Collisions Who Were Not Distracted',
        'Percentage Of Drivers Involved In Fatal Collisions Who Had Not Been Involved In Any Previous Accidents',
        'Car Insurance Premiums ($)', 'Losses incurred by insurance companies for collisions per insured driver ($)']
    if list(raw.columns) != expected_headers:
        raise RuntimeError('想定データ定義とCSV列が一致しない')
    df = raw.rename(columns=dict(zip(expected_headers, aliases)))
    years = [None, 2012, 2009, 2012, 2012, 2012, 2011, 2010]
    units = ['州名', '運転者/走行10億マイル', '%（死亡事故関与運転者内）', '%（死亡事故関与運転者内）', '%（死亡事故関与運転者内）', '%（死亡事故関与運転者内）', 'USD', 'USD/被保険運転者']
    variables = {}
    for alias, original, year, unit in zip(aliases, expected_headers, years, units):
        variables[alias] = {'definition': data_definition.FieldValue(original, 'verified', 'Kaggle CSVヘッダ・README'),
                            'unit': data_definition.FieldValue(unit, 'verified', 'Kaggle CSVヘッダ'),
                            'year': data_definition.FieldValue(year, 'verified' if year else 'unknown', 'Kaggle README'),
                            'measurement_detail': data_definition.FieldValue(None, 'unknown', 'READMEには具体的な集計・認定方法の記載なし')}
    variables['fatal_driver_rate']['exposure_denominator'] = data_definition.FieldValue('州内車両走行距離と解釈', 'inferred', 'per billion milesという列名。測定手順・車種範囲は未確認')
    definition_manifest = data_definition.build_manifest(
        source={'slug': data_definition.FieldValue(selected_ref, 'verified', 'Kaggle API'),
                'sha256': data_definition.FieldValue(provenance['sha256'], 'verified', '取得ファイル'),
                'retrieved_at': data_definition.FieldValue(provenance['retrieved_at_utc'], 'verified', '取得時記録'),
                'license': data_definition.FieldValue(None, 'unknown', '本実験では未確認')},
        dataset_scope={'coverage': data_definition.FieldValue('米国50州+District of Columbia', 'verified', 'CSVキー照合'),
                       'period_alignment': data_definition.FieldValue('2009/2010/2011/2012の異年指標', 'verified', 'README')},
        variables=variables, transformations=('列名を短縮。欠測補完・外れ値削除なし。州を等重みで比較。',))
    unknowns = definition_manifest.unresolved_fields()
    inferred = [(f'variables.{k}.{f}', dataclasses.asdict(v)) for k, fields in variables.items() for f, v in fields.items() if v.status == 'inferred']
    print('DATA_DEFINITION_MANIFEST', json.dumps(dataclasses.asdict(definition_manifest), ensure_ascii=False))
    print('UNRESOLVED_FIELDS', json.dumps([(key, dataclasses.asdict(value)) for key, value in unknowns], ensure_ascii=False))
    print('INFERRED_FIELDS_SUPPLEMENT', json.dumps(inferred, ensure_ascii=False))
    valid_states = set('Alabama|Alaska|Arizona|Arkansas|California|Colorado|Connecticut|Delaware|District of Columbia|Florida|Georgia|Hawaii|Idaho|Illinois|Indiana|Iowa|Kansas|Kentucky|Louisiana|Maine|Maryland|Massachusetts|Michigan|Minnesota|Mississippi|Missouri|Montana|Nebraska|Nevada|New Hampshire|New Jersey|New Mexico|New York|North Carolina|North Dakota|Ohio|Oklahoma|Oregon|Pennsylvania|Rhode Island|South Carolina|South Dakota|Tennessee|Texas|Utah|Vermont|Virginia|Washington|West Virginia|Wisconsin|Wyoming'.split('|'))
    if set(df['state']) != valid_states or len(df) != 51:
        raise RuntimeError('州の網羅性が不一致')
    schema = {'state': {'allowed': valid_states, 'not_null': True, 'unique': True},
              'fatal_driver_rate': {'min': 0, 'not_null': True},
              'premium_usd': {'min': 0, 'not_null': True}, 'loss_usd': {'min': 0, 'not_null': True}}
    schema.update({key: {'min': 0, 'max': 100, 'not_null': True} for key in aliases[2:6]})
    anomalies = data_quality.detect_anomalies(df, schema=schema)
    for col in aliases[1:]:
        if not pd.api.types.is_numeric_dtype(df[col]) or not np.isfinite(df[col]).all():
            raise RuntimeError('数値型または有限値の検査失敗: ' + col)
    print('SEMANTIC_ANOMALIES', json.dumps([dataclasses.asdict(a) for a in anomalies], ensure_ascii=False))
    if anomalies:
        raise RuntimeError('意味的な異常があるため確認なしに分析しない')
    cleaning_report = cleaning.clean_dataset(df, operation='drop_duplicates')
    df = cleaning_report.dataframe
    print('CLEANING_IMPACT', cleaning_report.rows_before, cleaning_report.rows_after, cleaning_report.rows_removed, cleaning_report.columns_affected)
    eda_report = eda.explore(df)
    print('EDA_REPORT', eda_report)
    assumptions_manifest = analysis_assumptions.AnalysisAssumptionManifest(
        analysis_scope={'target': '死亡事故関与運転者率と州別指標の関連', 'unit': '州/DC', 'time': '異年横断比較', 'weight': '州等重み'},
        causal_scope='associational', assumptions=(
            analysis_assumptions.Assumption('schema', '州網羅性・欠測・数値範囲を確認', 'verified'),
            analysis_assumptions.Assumption('equal-state', '全国の運転者リスクでなく典型的な州間の関係を推定', 'verified'),
            analysis_assumptions.Assumption('measurement', '州間で事故記録・注意散漫・飲酒の認定が比較可能', 'assumed', conclusion_critical=True, impact_if_false='州差に記録差が混在'),
            analysis_assumptions.Assumption('temporal', '異年指標が同時点の関係を代表する', 'assumed', conclusion_critical=True, impact_if_false='速度・保険の関連は同年の関係ではない'),
            analysis_assumptions.Assumption('causal', '交絡なく因果効果が識別できる', 'rejected', impact_if_false='因果効果は主張しない'),
            analysis_assumptions.Assumption('independence', 'p値計算で州を独立観測として扱う', 'assumed', conclusion_critical=True, impact_if_false='空間依存によりp値が過小評価され得る')))
    assumption_findings = analysis_assumptions.check_manifest(assumptions_manifest)
    print('ASSUMPTION_MANIFEST', json.dumps(dataclasses.asdict(assumptions_manifest), ensure_ascii=False))
    print('ASSUMPTION_FINDINGS', json.dumps([dataclasses.asdict(f) for f in assumption_findings], ensure_ascii=False))
'''
stats_code = '''from ai_data_scientist import stats_analysis
with phase('初回分析・州別順位と相関'):
    predictor_labels = {'speeding_pct': '速度超過割合（2009）', 'alcohol_pct': '飲酒割合（2012）',
                        'not_distracted_pct': '注意散漫でない割合（2012）', 'no_previous_pct': '事故歴なし割合（2012）',
                        'premium_usd': '保険料（2011）', 'loss_usd': '保険会社損失（2010）'}
    rows = []
    for feature, label in predictor_labels.items():
        result = stats_analysis.correlation(df, feature, 'fatal_driver_rate', language='ja')
        ranked = stats.spearmanr(df[feature], df['fatal_driver_rate'])
        rows.append({'feature': feature, 'label': label, 'pearson_r': result.statistic, 'p': result.p_value,
                     'spearman_rho': float(ranked.statistic), 'spearman_p': float(ranked.pvalue), 'interpretation': result.interpretation})
    correlations = pd.DataFrame(rows)
    def bh_adjust(p_values):
        p = np.asarray(p_values, dtype=float)
        order = np.argsort(p)
        adjusted = np.minimum.accumulate((p[order] * len(p) / np.arange(1, len(p)+1))[::-1])[::-1]
        out = np.empty(len(p))
        out[order] = np.minimum(adjusted, 1)
        return out
    correlations['q_bh'] = bh_adjust(correlations['p'])
    correlations['spearman_q_bh'] = bh_adjust(correlations['spearman_p'])
    print('初回相関（探索的、PearsonとSpearman各6比較のBH補正）')
    display(correlations.round(6))
    print('CORRELATIONS_JSON', correlations.to_json(orient='records', force_ascii=False, double_precision=12))
    print('単位: 死亡事故関与運転者/走行10億マイル。死亡者数や死亡事故件数ではない。')
    ranked_states = df[['state', 'fatal_driver_rate']].sort_values('fatal_driver_rate', ascending=False)
    print('州別全順位')
    display(ranked_states.reset_index(drop=True))
    print('RANGE_JSON', json.dumps({'n': len(df), 'min': float(df.fatal_driver_rate.min()), 'max': float(df.fatal_driver_rate.max()),
                                  'median': float(df.fatal_driver_rate.median()), 'equal_state_mean': float(df.fatal_driver_rate.mean()),
                                  'highest': ranked_states.head(5).to_dict('records'), 'lowest': ranked_states.tail(5).to_dict('records')}, ensure_ascii=False))
'''
charts_code = '''from ai_data_scientist import visualization
import warnings
with phase('初回可視化'):
    chart_specs = [
        {'data': df[['state', 'fatal_driver_rate']].sort_values('fatal_driver_rate', ascending=False).head(10), 'kind': 'bar', 'x': 'state', 'y': 'fatal_driver_rate', 'title': '死亡事故関与運転者率の上位10州（2012年）', 'xlabel': '州（州等重みの比較）', 'ylabel': '運転者 / 走行10億マイル'},
        {'data': df, 'kind': 'scatter', 'x': 'alcohol_pct', 'y': 'fatal_driver_rate', 'title': '飲酒割合と死亡事故関与運転者率（2012年）', 'xlabel': '死亡事故関与運転者中の飲酒割合 (%)', 'ylabel': '運転者 / 走行10億マイル'},
        {'data': df, 'kind': 'scatter', 'x': 'premium_usd', 'y': 'fatal_driver_rate', 'title': '保険料（2011年）と死亡事故関与運転者率（2012年）', 'xlabel': '保険料 (USD)', 'ylabel': '運転者 / 走行10億マイル'}]
    chart_audit_metadata = []
    for spec in chart_specs:
        with warnings.catch_warnings(record=True) as captured:
            warnings.simplefilter('always')
            png = visualization.render_chart(spec['data'], **{k: v for k, v in spec.items() if k != 'data'})
        glyph_warnings = [str(w.message) for w in captured if 'Glyph' in str(w.message)]
        metadata = {k: spec[k] for k in ['title', 'xlabel', 'ylabel']}
        metadata.update({'legend': '単系列。凡例は不要（軸ラベルで識別）', 'missing_glyphs': glyph_warnings, 'png_bytes': len(png)})
        chart_audit_metadata.append(metadata)
        image_output = visualization.build_image_output(png)
        display({'image/png': image_output.data['image/png'], 'text/plain': spec['title']}, raw=True)
    print('CHART_METADATA', json.dumps(chart_audit_metadata, ensure_ascii=False))
'''
caveats = '''## 比較対象と分析前提の解釈
一般の交通事故率、人口当たり死亡率、死亡事故が起きたときの致死率は、このCSVから直接算出できない。得られるのは死亡事故に関与した運転者の走行距離当たり率と、すでに死亡事故に関与した運転者の特徴割合である。飲酒割合等は全運転者の曝露割合ではない。複数の特徴は重なり得るので足して100%にしない。「注意散漫でない」「事故歴なし」の符号を逆転して読まない。

走行距離による正規化は州人口や交通量の差を一部調整するが、都市化、道路構造、走行速度、天候、車種、救急アクセス、認定・記録方法を揃えない。州を等重みとし、DCも初回には含める。全国合計率や運転者個人の因果効果への一般化はしない。

前提manifestの全警告（measurement / temporal / independence）は未解決であり、州間の測定比較可能性、2009～2012年の異年混在、空間依存を仮定している。p値・BH補正は探索の補助で、州全数の記述に無作為標本としての厳密な意味は付与しない。因果識別は不成立。変数のmeasurement_detailとlicenseはunknown、走行距離の測定・車種範囲はinferredとして記録する。

独立参照データのvalidate_anomaliesとcompare_datasetsは非適用。同一FiveThirtyEightデータの別Kaggleコピーは独立検証にならず、独立した同年・同一定義データも供給されていない。時系列予測、人口/走行距離重み付け、交絡調整の本格的因果分析も必要列がないため非適用。
'''
append_cells([nbformat.v4.new_code_cell(definitions), nbformat.v4.new_markdown_cell(caveats), nbformat.v4.new_code_cell(stats_code), nbformat.v4.new_code_cell(charts_code)])
print('定義・前提・初回分析・図表コードを記録')

定義・前提・初回分析・図表コードを記録


In [17]:
from ai_data_scientist import insight_engine
current_nb = nbformat.read(handle.notebook_path, as_version=4)
def add_insight_from(index, pattern, text, claim='associational'):
    evidence_cell = nbformat.read(handle.notebook_path, as_version=4).cells[index]
    output_text = '\n'.join(str(value) for output in evidence_cell.outputs for value in output.get('data', {}).values())
    cited = insight_engine.extract_cited_value({'output': output_text}, pattern)
    lifecycle.mark_write_start(RUN_ID)
    try:
        insight_engine.record_insight(handle, text, evidence_cell.execution_count, cited, claim, language='ja')
        def tag(nb):
            nb.cells[-1].metadata['evidence_source_cell_id'] = evidence_cell.id
        project_manager.enqueue_write(handle, tag)
    finally:
        lifecycle.mark_write_end(RUN_ID)
add_insight_from(8, r'(0\.199426)', '初回Insight：飲酒割合と死亡事故関与運転者率のPearson相関は約0.199、Spearmanは約0.051で弱い。6変数すべてBH補正後q>0.48で、ここでは明確な州間の単変量関連は検出されなかった。飲酒が安全、または影響がないという意味ではない。')
add_insight_from(8, r'(23\.9)', '初回Insight：North DakotaとSouth Carolinaは23.9、West Virginiaは23.8で高く、DCは5.9、Massachusettsは8.2で低い。これは走行10億マイル当たり死亡事故関与運転者率の比較であり、事故件数・死亡者率の順位ではない。', 'descriptive')
interpretations = '\n\n'.join(row['label'] + '：' + row['interpretation'] for row in correlations.to_dict('records'))
append_cells([nbformat.v4.new_markdown_cell('## 初回統計モジュールの解釈\n' + interpretations + '\n\n上記は未補正p値による自動文。実質的な結論はBH補正・散布図・感度分析と併せて判断する。')])
# 描画結果の目視で長い州名の下端が切れていたため、横棒と余白調整に変更する。
def improve_chart(nb):
    cell = nb.cells[9]
    cell.source = cell.source.replace('import warnings', 'import warnings\nimport matplotlib.pyplot as plt')
    old = "            png = visualization.render_chart(spec['data'], **{k: v for k, v in spec.items() if k != 'data'})"
    new = """            if spec['kind'] == 'bar':
                fig, ax = plt.subplots(figsize=(10, 6))
                ordered = spec['data'].sort_values('fatal_driver_rate')
                ax.barh(ordered['state'], ordered['fatal_driver_rate'])
                ax.set(title=spec['title'], xlabel=spec['ylabel'], ylabel='州')
                fig.tight_layout()
                buffer = io.BytesIO()
                fig.savefig(buffer, format='png', bbox_inches='tight')
                plt.close(fig)
                png = buffer.getvalue()
            else:
                png = visualization.render_chart(spec['data'], **{k: v for k, v in spec.items() if k != 'data'})"""
    cell.source = cell.source.replace(old, new)
    cell.source = cell.source.replace("        metadata.update({'legend':", "        if spec['kind'] == 'bar':\n            metadata['xlabel'], metadata['ylabel'] = spec['ylabel'], '州'\n        metadata.update({'legend':")
    cell.metadata['chart'] = {'title': '上位10州・飲酒割合・保険料の3図', 'xlabel': '各図の軸に単位を明示', 'ylabel': '各図の軸に単位を明示', 'legend': '単系列/点群。凡例または軸ラベルで識別', 'missing_glyphs': []}
    cell.outputs = []
    cell.execution_count = None
lifecycle.mark_write_start(RUN_ID)
try:
    project_manager.enqueue_write(handle, improve_chart)
finally:
    lifecycle.mark_write_end(RUN_ID)
request1 = '初回の相関が弱いという結論は、DCの特殊性、一部の州、直線性の仮定に依存していませんか。DCを除外し、PearsonとSpearmanを比較し、全州のleave-one-outと固定seedの州単位ブートストラップで不確実性を確認してください。弱い相関付近では相対偏差が大きくなる点も分けて説明してください。'
iteration1 = '''from ai_data_scientist import sensitivity
with phase('反復1: DC・相関仕様・単一州依存・区間推定'):
    sensitivity_reports = {}
    sensitivity_rows = []
    loo_rows = []
    bootstrap_rows = []
    rng = np.random.default_rng(20261002)
    for feature in predictor_labels:
        plan = sensitivity.SensitivityPlan(parameter_grid={'exclude_dc': [False, True], 'method': ['pearson', 'spearman']}, max_runs=4)
        def spec_correlation(exclude_dc, method):
            subset = df[df.state != 'District of Columbia'] if exclude_dc else df
            return float(subset[feature].corr(subset['fatal_driver_rate'], method=method))
        report = sensitivity.run_sensitivity(plan, spec_correlation, stability_tolerance=0.20)
        sensitivity_reports[feature] = report
        for item in report.results:
            sensitivity_rows.append({'feature': feature, **item.specification, 'r': item.value,
                                     'stable': report.stable, 'max_relative_deviation': report.max_relative_deviation})
        loo = [(state, float(df[df.state != state][feature].corr(df[df.state != state]['fatal_driver_rate']))) for state in df.state]
        loo_rows.append({'feature': feature, 'loo_min': min(v for _,v in loo), 'loo_max': max(v for _,v in loo),
                         'max_change_state': max(loo, key=lambda item: abs(item[1]-report.baseline_value))[0]})
        for exclude_dc in [False, True]:
            subset = df[df.state != 'District of Columbia'] if exclude_dc else df
            x = subset[feature].to_numpy(float)
            y = subset.fatal_driver_rate.to_numpy(float)
            samples = rng.integers(0, len(subset), size=(4000, len(subset)))
            bx, by = x[samples], y[samples]
            cx, cy = bx-bx.mean(axis=1, keepdims=True), by-by.mean(axis=1, keepdims=True)
            boot_r = (cx*cy).sum(axis=1) / np.sqrt((cx*cx).sum(axis=1)*(cy*cy).sum(axis=1))
            if not np.isfinite(boot_r).all():
                raise RuntimeError('ブートストラップに未定義相関')
            low, high = np.quantile(boot_r, [0.025, 0.975])
            bootstrap_rows.append({'feature': feature, 'exclude_dc': exclude_dc, 'bootstrap_low': float(low), 'bootstrap_high': float(high)})
    sensitivity_table = pd.DataFrame(sensitivity_rows)
    loo_table = pd.DataFrame(loo_rows)
    bootstrap_table = pd.DataFrame(bootstrap_rows)
    print('感度計画: 6変数×4仕様（各変数max_runs=4）、相対偏差許容0.20')
    print('州ブートストラップ: 4000回/変数/対象、seed=20261002。空間独立を仮定し、全国率CIではない。')
    display(sensitivity_table.round(6))
    display(loo_table.round(6))
    display(bootstrap_table.round(6))
    sensitivity_summary = {'max_abs_spec_r': float(sensitivity_table.r.abs().max()), 'max_abs_loo_r': float(loo_table[['loo_min','loo_max']].abs().max().max()),
                           'all_bootstrap_intervals_include_zero': bool(((bootstrap_table.bootstrap_low <= 0)&(bootstrap_table.bootstrap_high >= 0)).all()),
                           'stable_by_feature': {key: value.stable for key,value in sensitivity_reports.items()},
                           'max_relative_deviation_by_feature': {key: value.max_relative_deviation for key,value in sensitivity_reports.items()}}
    display({'text/plain': 'ITERATION1_SUMMARY ' + json.dumps(sensitivity_summary, ensure_ascii=False)}, raw=True)
'''
append_cells([nbformat.v4.new_markdown_cell('## 反復依頼履歴\n### 反復1：原文\n' + request1 + '\n\n選定理由：初回に弱い相関しかなかったため、DCや特定州を含めることが結論を左右していないかを優先検証する。仕様の安定性、実質的な大きさ、区間の広さを分けて判断する。'), nbformat.v4.new_code_cell(iteration1)])
print('初回Insight、図表修正、反復1の依頼原文とコードを記録')

初回Insight、図表修正、反復1の依頼原文とコードを記録


In [18]:
def font_fix(nb):
    nb.cells[9].source = nb.cells[9].source.replace('import matplotlib.pyplot as plt', 'import matplotlib.pyplot as plt\nimport japanize_matplotlib\njapanize_matplotlib.japanize()')
lifecycle.mark_write_start(RUN_ID)
try:
    project_manager.enqueue_write(handle, font_fix)
finally:
    lifecycle.mark_write_end(RUN_ID)
print('横棒図も新規カーネルで日本語フォントが適用されるように設定')

横棒図も新規カーネルで日本語フォントが適用されるように設定


In [21]:
add_insight_from(14, r'("max_abs_loo_r": [0-9.]+)', '反復1 Insight：DC有無×Pearson/Spearmanの点推定は全て|r|<0.20、leave-one-outは最大|r|約0.268。中程度の関連（探索上の目安|r|>=0.30）を示す点推定はなかった。全12ブートストラップ区間は0を含むが、中程度の関連を完全には排除しない。sensitivityのstableは全6変数Falseで、相対偏差は約0.408～15.367。ゼロ付近の係数では相対尺度が不安定なため「効果ゼロが頑健」とは主張しない。')
request2 = '割合では相関が弱い一方、飲酒関与の死亡事故運転者率に変換すると強い相関が現れませんか。2012年の同年指標に限定して、総率×飲酒割合から派生率を作り、総率が両方に入る数学的結合を固定割合、共分散分解、割合の置換実験で点検してください。派生率の強い相関を飲酒の因果効果や独立した危険因子の証拠と誤解しない説明と図を追加してください。'
iteration2 = '''with phase('反復2: 同年指標・派生率の数学的結合'):
    same_year_features = ['alcohol_pct', 'not_distracted_pct', 'no_previous_pct']
    same_year = correlations[correlations.feature.isin(same_year_features)].copy()
    same_year['q_bh_2012'] = bh_adjust(same_year['p'])
    display(same_year[['feature','pearson_r','p','q_bh_2012']].round(6))
    rate = df.fatal_driver_rate.to_numpy(float)
    alcohol_fraction = df.alcohol_pct.to_numpy(float) / 100
    derived_rate = rate * alcohol_fraction
    derived_df = df.assign(alcohol_driver_rate=derived_rate)
    if (derived_df.alcohol_driver_rate < 0).any() or (derived_df.alcohol_driver_rate > derived_df.fatal_driver_rate).any():
        raise RuntimeError('派生率が意味的範囲に違反')
    derived_variables = dict(definition_manifest.variables)
    derived_variables['alcohol_driver_rate'] = {
        'definition': data_definition.FieldValue('死亡事故関与運転者率×飲酒割合/100', 'verified', '本Notebookの算式'),
        'unit': data_definition.FieldValue('飲酒関与運転者/走行10億マイル', 'inferred', '同年2012の母集団一致を仮定'),
        'population_alignment': data_definition.FieldValue('率と割合の分子母集団・測定が一致', 'inferred', 'READMEで詳細未確認。丸めの影響あり')}
    derived_manifest = data_definition.build_manifest(definition_manifest.source, definition_manifest.dataset_scope, derived_variables,
        transformations=definition_manifest.transformations + ('alcohol_driver_rate = fatal_driver_rate * alcohol_pct / 100',))
    print('DERIVED_MANIFEST', json.dumps(dataclasses.asdict(derived_manifest), ensure_ascii=False))
    cov_observed = float(np.mean((rate-rate.mean())*(derived_rate-derived_rate.mean())))
    structural_cov = float(alcohol_fraction.mean() * np.var(rate))
    residual_cov = float(np.mean((rate-rate.mean())*(rate*(alcohol_fraction-alcohol_fraction.mean()))))
    if not np.isclose(cov_observed, structural_cov+residual_cov):
        raise RuntimeError('共分散分解が一致しない')
    observed_r = float(np.corrcoef(rate, derived_rate)[0,1])
    fixed_fraction_r = float(np.corrcoef(rate, rate*alcohol_fraction.mean())[0,1])
    rng2 = np.random.default_rng(20261003)
    permutation_r = np.array([np.corrcoef(rate, rate*rng2.permutation(alcohol_fraction))[0,1] for _ in range(5000)])
    permutation_interval = np.quantile(permutation_r, [0.025, 0.975])
    coupling_summary = {'original_percentage_r': float(np.corrcoef(rate, alcohol_fraction)[0,1]),
                        'derived_rate_r': observed_r, 'fixed_fraction_r': fixed_fraction_r,
                        'coupled_permutation_mean_r': float(permutation_r.mean()),
                        'coupled_permutation_95_low': float(permutation_interval[0]), 'coupled_permutation_95_high': float(permutation_interval[1]),
                        'upper_tail_probability': float((1+(permutation_r >= observed_r).sum())/(len(permutation_r)+1)),
                        'cov_observed': cov_observed, 'structural_cov': structural_cov, 'residual_cov': residual_cov,
                        'structural_cov_fraction': structural_cov/cov_observed,
                        'permutations': 5000, 'seed': 20261003}
    display({'text/plain': 'ITERATION2_SUMMARY ' + json.dumps(coupling_summary, ensure_ascii=False)}, raw=True)
    coupling_plan = sensitivity.SensitivityPlan(parameter_grid={'exclude_dc': [False, True], 'method': ['pearson','spearman']}, max_runs=4)
    def derived_spec(exclude_dc, method):
        subset = derived_df[derived_df.state != 'District of Columbia'] if exclude_dc else derived_df
        return float(subset.fatal_driver_rate.corr(subset.alcohol_driver_rate, method=method))
    coupling_sensitivity = sensitivity.run_sensitivity(coupling_plan, derived_spec, stability_tolerance=0.2)
    print('派生率の数値的安定性（因果・独立性を保証しない）:', dataclasses.asdict(coupling_sensitivity))
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    axes[0].scatter(alcohol_fraction*100, rate)
    axes[0].set(title='元の飲酒割合との関連', xlabel='死亡事故関与運転者中の飲酒割合 (%)', ylabel='総運転者率 / 走行10億マイル')
    axes[1].scatter(derived_rate, rate)
    axes[1].set(title='総率を含む派生率との関連', xlabel='飲酒関与運転者の派生率 / 走行10億マイル', ylabel='総運転者率 / 走行10億マイル')
    axes[2].hist(permutation_r, bins=40, label='割合を置換しても残る相関')
    axes[2].axvline(observed_r, color='red', label='観測された派生率相関')
    axes[2].set(title='数学的結合を残した置換分布', xlabel='総率と派生率のPearson相関', ylabel='置換回数')
    axes[2].legend()
    fig.tight_layout()
    buffer = io.BytesIO()
    with warnings.catch_warnings(record=True) as warnings2:
        warnings.simplefilter('always')
        fig.savefig(buffer, format='png', bbox_inches='tight')
    plt.close(fig)
    coupling_glyph_warnings = [str(w.message) for w in warnings2 if 'Glyph' in str(w.message)]
    if coupling_glyph_warnings:
        raise RuntimeError('派生率図で文字欠損を検出')
    coupling_png = buffer.getvalue()
    coupling_output = visualization.build_image_output(coupling_png)
    display({'image/png': coupling_output.data['image/png'], 'text/plain': '同年2012: 割合/派生率/数学的結合'}, raw=True)
'''
append_cells([nbformat.v4.new_markdown_cell('## 反復依頼履歴\n### 反復2：原文\n' + request2 + '\n\n選定理由：割合の弱い相関と、よく用いられる部分率の強い相関が見かけ上矛盾し得る。説明変数に目的変数が混ざる点は結論に実質的な影響があるため、同年2012の飲酒指標に限定して検証する。置換は因果効果検定でなく、数学的結合の診断として用いる。'), nbformat.v4.new_code_cell(iteration2)])
def set_coupling_meta(nb):
    nb.cells[-1].metadata['chart'] = {'title':'飲酒割合・派生率・数学的結合の3パネル', 'xlabel':'各パネルに割合・率・相関の単位を明示', 'ylabel':'率/置換回数', 'legend':'第3パネルで観測値と置換分布を識別', 'missing_glyphs': []}
lifecycle.mark_write_start(RUN_ID)
try:
    project_manager.enqueue_write(handle, set_coupling_meta)
finally:
    lifecycle.mark_write_end(RUN_ID)
print('反復1の根拠付き結果と反復2の原文・コードを記録')

反復1の根拠付き結果と反復2の原文・コードを記録


In [24]:
add_insight_from(17, r'("derived_rate_r": [0-9.]+)', '反復2 Insight：2012年の飲酒割合ではr≈0.199だが、総率×飲酒割合の派生率ではr≈0.853になる。割合を固定するとr=1、割合を置換しても平均r≈0.835（中央95%範囲0.778～0.879）で、観測値はこの範囲内。共分散の約85.8%は平均割合×総率分散の項。派生率の相関の強さは独立した危険因子・因果効果の証拠にならない。DC有無・Pearson/Spearmanでは派生相関のstable=True、最大相対偏差≈0.0497だが、数値的安定性は解釈の妥当性を保証しない。')
summary = '''## 日本語報告：結論と反復の評価

### 何を測定できたか
Kaggle `fivethirtyeight/fivethirtyeight-bad-drivers-dataset` の `bad-drivers.csv`、50州＋DC（51行）を分析した。CSV取得は2026-10-01 UTC（日本時間2026-10-02）。来歴セルにowner/slug・ファイル名・取得時刻・SHA-256を記録した。一般事故率、死亡者率、人口当たり死亡率ではなく、2012年の「死亡事故関与運転者/走行10億マイル」が対象。現時点の安全状況ではない。

### 州間比較と特徴の関連
最高はNorth DakotaとSouth Carolinaの23.9、West Virginiaの23.8。最低はDCの5.9、州のみではMassachusettsの8.2。中央値は15.6。欠測・重複・意味的制約違反は検出されず、補完も外れ値除外も行っていない。

| 特徴 | 年 | Pearson r | Spearman rho | Pearson BH q |
|---|---:|---:|---:|---:|
| 速度超過割合 | 2009 | -0.029 | -0.017 | 0.946 |
| 飲酒割合 | 2012 | 0.199 | 0.051 | 0.482 |
| 注意散漫でない割合 | 2012 | 0.010 | 0.093 | 0.946 |
| 事故歴なし割合 | 2012 | -0.018 | 0.019 | 0.946 |
| 保険料 | 2011 | -0.200 | -0.118 | 0.482 |
| 保険会社損失 | 2010 | -0.036 | ほぼ0 | 0.946 |

単変量の州間関連は弱く、6比較のBH補正で明確な関連を検出しなかった。保険料の負相関を保険料の保護効果と読むことも、速度・飲酒・注意散漫が安全だと読むこともできない。表の根拠は初回統計セルと隣接するevidence付きInsight。

### 反復1の結果・証拠・限界
選定理由はDC・外れ州・直線性への依存の確認。24仕様の|r|は最大約0.200、leave-one-outの最大|r|は約0.268。全12個の州ブートストラップ区間は0を含む。飲酒の全州区間は約[-0.163, 0.478]であり、関連がないと確定するほど狭くない。相対偏差20%基準では全6変数stable=False、最大相対偏差は変数別約0.408～15.367。ゼロ近傍の相対偏差は膨らむため、弱い点推定が一貫していることと符号・係数の安定性を区別する。根拠：反復1セルの表、ITERATION1_SUMMARYとevidence付きInsight。残る限界：小標本、州の空間依存、地域・道路・記録方法の交絡を扱えていない。

### 反復2の結果・証拠・限界
選定理由は割合と派生率の混同が結論を変えるため。同年2012の3特徴だけでもBH補正qは最小約0.482。飲酒の派生率と総率のrは約0.853だが、割合置換5000回の結合を残した分布は平均約0.835、中央95%範囲約[0.778, 0.879]で、観測r以上の比率は約0.262。固定割合ならr=1。共分散分解の構造項割合は約85.8%だが、これは説明した因果割合・R²ではない。根拠：ITERATION2_SUMMARY、共分散検算、3パネル図、evidence付きInsight。派生率は丸められた2列を掛けた近似で、分子母集団の完全一致は未確認。置換は数学的結合の診断で、飲酒の個人リスクについての検定ではない。

### 因果・比較可能性の制限と終了判断
走行距離正規化だけでは道路、都市化、天候、車種、救急アクセス、州別認定手順を揃えられない。事故に関与した運転者だけの割合には選択・構成比の問題があり、全運転者の曝露差も不明。集計相関から個人に関する因果関係は言えない（生態学的誤謬）。単年の横断集計では交絡も逆因果も除去できず、速度2009、損失2010、保険料2011、死亡関連2012の年差もある。measurement / temporal / independenceの前提警告は全て未解決。

反復は2回実施して停止する。3回目は行わない。残る重要点は同年の人口・走行距離・道路条件・全運転者の曝露・記録方法を持つ独立データを必要とし、現CSVだけで追加モデルや探索を増やしても識別が改善しないため。外部ソースへフォールバックせず、価値の低い追加検定を避ける。
'''
improvements = '''## Jupytermind改善候補

| 分類 | 再現条件 | 期待する挙動 | 実際の挙動 | 影響 | 回避策 | 関係するセルとログ |
|---|---|---|---|---|---|---|
| 可視化・レイアウト機能不足 | visualization.render_chartで上位10州、kind=bar、x=長い州名、y=率、日本語ラベル | 軸ラベル・州名が画像領域内に収まる、またはfigsize/tight_layout指定が可能 | 初回縦棒の州名下端が切れた。render_chartは固定figsize・余白調整引数なし | 州比較の読み取りが難しい | 横棒図をMatplotlibで作りtight_layoutとbbox_inches=tightを適用 | 初回可視化セル（index 9）の修正前MCP画像SHA先頭0d54a4b3c70f0c2f、修正後f4a2c0704e3d6a7d。Notebook記録診断も参照 |
| データ定義・リスク検出の仕様不足 | FieldValue.status=inferredを含むmanifest.unresolved_fields() | unknownに加え未検証inferredも一覧化、または別APIで区別して返す | unknownのみが返る（実装のdocstring通りであり契約違反とは断定しない）。Skill手順が求めるinferred caveatは自動抽出されない | 推測した分母・派生率母集団が未解決一覧から漏れる | status=inferredを別途列挙し、Notebookの caveat に明示 | 定義manifestセル（index 6）のINFERRED_FIELDS_SUPPLEMENT、改善候補再現ログINFERRED_NOT_RETURNED |
| 図表監査・メタデータ欠如の見逃し | build_image_outputで作った非空PNGをchartメタデータなしのセルに置きaudit_visual_outputsを実行 | title/xlabel/ylabel/legend/glyph監査情報の欠如を警告 | chart metadata={}の場合はラベル検査自体をスキップし空の結果になる | visual_audit=Trueだけではタイトル・軸・文字監査の完了を保証しない | 実図のラベルと文字警告を検査し、セルmetadata.chartと図ごとのCHART_METADATAをenqueue_writeで付与 | 改善候補再現ログNO_METADATA_VISUAL_FINDINGS、初回図・反復2図・最終監査 |

上記はJupytermindの直接API呼び出しと出力に基づく機能不足の候補で、修正は本分析の対象外。SDK dataset_listのpage_size TypeErrorはKaggle API固有であり上表に含めない。Jupyter MCP use_notebookが親ディレクトリなしで作成できなかった件はMCPの操作条件で、project_manager.ensure_notebookで作成して解消。Copilot CLI・ベンチマークランナー固有の問題はこの実験では確認されていない。

mcp_gateway.run_and_recordはNotebook内から同一カーネルへMCPクライアントを再帰接続するため非適用（直接実行はJupyter MCPに限定）。この環境上の適用制限をJupytermind不具合と混同しない。Insightの参照には実出力のexecution_countを用い、再実行後はセルIDから参照を更新する。
'''
modules = '''## 使用したJupytermindモジュール
直接import・直接呼び出しのみを列挙する。操作用bootstrapセルでの直接利用も区別して記録し、間接利用を使用済みとして数えない。

| モジュール | 直接呼び出した関数・クラス | 使用目的・場所 |
|---|---|---|
| ai_data_scientist.project_manager | resolve_project, ensure_notebook, ensure_data_dir, enqueue_write | 安定した保存先・データディレクトリ、Notebookの作成・全追加/編集の直列化（分析セル・操作セル） |
| ai_data_scientist.language_router | detect_language | 日本語選択（設定セル） |
| ai_data_scientist.lifecycle | register_run, mark_execution_start, mark_execution_end, mark_write_start, mark_write_end, is_cancel_requested, mark_completed, mark_failed, get_run_status, wait_for_quiescence | run_id登録、実行/書込みフェーズ、失敗・完了・静穏状態（設定・各分析・操作・最終セル） |
| ai_data_scientist.ingestion | SourceSpec, ingest | Kaggle SDKで取得したローカルCSVのロード（取得セル） |
| ai_data_scientist.cleaning | clean_dataset | 重複確認と除去の影響記録（品質セル、除去0行） |
| ai_data_scientist.eda | explore | 型・欠測・記述統計・州名カテゴリ確認（品質セル） |
| ai_data_scientist.stats_analysis | correlation | 日本語解釈付きPearson相関（初回統計セル） |
| ai_data_scientist.visualization | render_chart, build_image_output | 日本語散布図の描画、PNG MIME出力の組立て（図表セル） |
| ai_data_scientist.insight_engine | extract_cited_value, record_insight | 実出力から引用値を抽出し、検証済みevidenceを付ける（操作用bootstrapセル）。最終セルではextract_cited_valueを直接使用 |
| ai_data_scientist.data_definition | FieldValue, build_manifest, DataDefinitionManifest.unresolved_fields | 定義・単位・年・来歴と未解決項目。派生率の推測状態の記録 |
| ai_data_scientist.analysis_assumptions | Assumption, AnalysisAssumptionManifest, check_manifest | 比較可能性・異年指標・因果スコープの明示 |
| ai_data_scientist.data_quality | detect_anomalies | 州キー、非欠測、範囲、唯一性の意味的検査 |
| ai_data_scientist.sensitivity | SensitivityPlan, run_sensitivity | DC有無・Pearson/Spearmanの4仕様を2種類の相関で検査 |
| ai_data_scientist.notebook_audit | audit_visual_outputs, audit_notebook | 改善候補の再現とvisual_audit=Trueの最終監査 |

未使用：mcp_gateway（直接MCPを利用する環境上の非適用）、dataset_validation / validate_anomalies（独立参照データなし）、anomaly_detection（意味的制約とleave-one-outを優先）。record_chartはレンダリング済みセルを実行済み扱いで記録するが実カーネル実行ではないため使用せず、図表コードを実際にJupyter MCPで実行した。横棒・3パネル図・Bootstrap・置換はMatplotlib/NumPy/SciPyを直接使用しており、Jupytermindの提供機能として数えない。
'''
diagnostics = '''from ai_data_scientist import notebook_audit
with phase('改善候補の再現ログ・適用範囲確認'):
    inferred_paths = [f'variables.{k}.{f}' for k,fields in definition_manifest.variables.items() for f,v in fields.items() if v.status == 'inferred']
    unresolved_paths = [path for path,_ in definition_manifest.unresolved_fields()]
    print('INFERRED_NOT_RETURNED', sorted(set(inferred_paths)-set(unresolved_paths)))
    probe_cell = nbformat.v4.new_code_cell('visualization.build_image_output(coupling_png)')
    probe_cell.outputs = [visualization.build_image_output(coupling_png)]
    probe_nb = nbformat.v4.new_notebook(cells=[probe_cell])
    missing_meta_findings = notebook_audit.audit_visual_outputs(probe_nb, (0,))
    print('NO_METADATA_VISUAL_FINDINGS', [dataclasses.asdict(f) for f in missing_meta_findings])
    print('NO_METADATA_PRESENT', dict(probe_cell.metadata))
    print('render_chart公開シグネチャ', str(inspect.signature(visualization.render_chart)))
    print('本成果物では各図のラベル・glyph warningsとmetadata.chartを確認して回避している。')
'''
validation = '''with phase('再現性チェックポイント'):
    snapshot = {'data_sha256': hashlib.sha256(original_path.read_bytes()).hexdigest(),
                'n': len(df), 'missing': int(df.isna().sum().sum()),
                'correlations': correlations[['feature','pearson_r','p','spearman_rho','q_bh']].to_dict('records'),
                'sensitivity': sensitivity_summary, 'coupling': coupling_summary,
                'bootstrap': bootstrap_table.to_dict('records'), 'loo': loo_table.to_dict('records')}
    snapshot_text = json.dumps(snapshot, ensure_ascii=False, sort_keys=True, separators=(',', ':'))
    baseline_path = data_dir / 'analysis-baseline.json'
    if baseline_path.exists():
        baseline_text = baseline_path.read_text(encoding='utf-8')
        if json.loads(baseline_text) != snapshot:
            raise RuntimeError('再実行結果が初回分析のベースラインと一致しない')
        verification_mode = 'replay_matched'
    else:
        baseline_path.write_text(snapshot_text, encoding='utf-8')
        verification_mode = 'baseline_recorded'
    snapshot_sha256 = hashlib.sha256(snapshot_text.encode('utf-8')).hexdigest()
    display({'text/plain': json.dumps({'verification_mode': verification_mode, 'snapshot_sha256': snapshot_sha256, 'data_sha256': snapshot['data_sha256'], 'n': len(df)}, ensure_ascii=False)}, raw=True)
'''
final_code = '''from ai_data_scientist import insight_engine
import re
with phase('根拠参照更新とNotebook監査'):
    # 再実行で変わったexecution_countを、変更しないセルIDから引き直す。
    persisted = nbformat.read(handle.notebook_path, as_version=4)
    by_id = {cell.id: cell for cell in persisted.cells}
    refreshed_sources = {}
    for cell in persisted.cells:
        source_id = cell.metadata.get('evidence_source_cell_id')
        if not source_id:
            continue
        source_cell = by_id[source_id]
        evidence_match = re.search(r'```evidence\\n(.*?)\\n```', cell.source, flags=re.S)
        evidence = json.loads(evidence_match.group(1))
        actual_output = '\\n'.join(str(value) for output in source_cell.outputs for value in output.get('data', {}).values())
        evidence['cited_value'] = insight_engine.extract_cited_value({'output': actual_output}, re.escape(evidence['cited_value']))
        evidence['execution_count'] = source_cell.execution_count
        replacement = '```evidence\\n' + json.dumps(evidence, separators=(',', ':')) + '\\n```'
        refreshed_sources[cell.id] = cell.source[:evidence_match.start()] + replacement + cell.source[evidence_match.end():]
    def refresh(nb):
        for cell in nb.cells:
            if cell.id in refreshed_sources:
                cell.source = refreshed_sources[cell.id]
        nb.metadata['analysis_run'] = {'run_id': RUN_ID, 'verification_mode': verification_mode, 'snapshot_sha256': snapshot_sha256, 'data_sha256': provenance['sha256']}
    lifecycle.mark_write_start(RUN_ID)
    try:
        project_manager.enqueue_write(handle, refresh)
    finally:
        lifecycle.mark_write_end(RUN_ID)
    audit_report = notebook_audit.audit_notebook(handle.notebook_path, visual_audit=True)
    print('NOTEBOOK_AUDIT', json.dumps({'ok': audit_report.ok, **dataclasses.asdict(audit_report)}, ensure_ascii=False))
    if not audit_report.ok:
        raise RuntimeError('Notebook監査で未解決のエラー')
lifecycle.mark_completed(RUN_ID)
final_status = lifecycle.wait_for_quiescence(RUN_ID, timeout_s=5)
print('FINAL_LIFECYCLE', json.dumps(dataclasses.asdict(final_status), ensure_ascii=False))
if final_status.state != 'completed' or final_status.active_cell_executions or final_status.pending_notebook_writes or final_status.locks_held:
    lifecycle.mark_failed(RUN_ID)
    raise RuntimeError('完了または静穏状態の検証失敗')
print('再現性モード:', verification_mode, '実行結果SHA-256:', snapshot_sha256)
'''
append_cells([nbformat.v4.new_markdown_cell(summary), nbformat.v4.new_markdown_cell(improvements), nbformat.v4.new_markdown_cell(modules), nbformat.v4.new_code_cell(diagnostics), nbformat.v4.new_markdown_cell('## 再実行確認・最終監査\n初回のデータ・統計・ブートストラップ・感度・置換結果をJSONベースラインへ記録し、新規カーネルで全コードセルを上から実行して完全一致を確認する。最終セルでevidenceの参照を実際の新execution_countへ更新し、visual_audit=Trueとlifecycleのcompleted/静穏を記録する。監査中の自己セルは未実行判定から除外される場合があるため、終了後に保存済みNotebookを読み直して全セル・画像・根拠を再確認する。'), nbformat.v4.new_code_cell(validation), nbformat.v4.new_code_cell(final_code)])
print('報告・改善候補・直接使用モジュール・再現性・監査セルを記録')

報告・改善候補・直接使用モジュール・再現性・監査セルを記録
